## Opening files

In [2]:
import pandas as pd

In [3]:
# queries

query_df = pd.read_xml("../data/kid-friend-en/en/inputs/topics.xml")
query_df.drop_duplicates(inplace=True, ignore_index=True)
query_df = query_df.rename({"number":"qid"}, axis=1)
query_df["query"] = ["".join([x if x.isalnum() else " " for x in query]) for query in query_df["query"]] # removing punctuations
query_df["qid"] = query_df["qid"].astype(str)
query_df.to_csv("../data/kid-friend-en/en/inputs/topics.csv", index=False)
query_df.head()

,qid,query,category,description,narrative
0,1,BTS,entertainment,Who are BTS? Where can I find their music?,Relevant documents for this request should inc...
1,2,BTS Bias Quiz,entertainment,Which band member suits me best? Which band me...,Relevant documents include kid-friendly conten...
2,3,BTS merch,entertainment,official BTS merch page and authorized sellers,Relevant documents redirect to fan articles of...
3,4,BTS Jungkook,entertainment,Who is Jungkook? What does he like? What does ...,Relevant documents contain information about J...
4,5,Minecraft,entertainment,What is Minecraft? Where can I play it? Let's ...,Relevant documents contain information about t...


In [18]:
# graded topical relevance

with open("../data/kid-friend-en/en/qrels/qrels-relevance.txt", "r") as f_qrel:
    lines= f_qrel.readlines()

qrels = []
for line in lines:
    qid, q0, docno, relevance = line.split(" ")
    qid = int(qid)
    relevance = int(relevance)
    qrels.append([qid, q0, docno, relevance])

qrels_df = pd.DataFrame(qrels, columns=["qid", "q0", "docno", "relevance"])
qrels_df["qid"] = qrels_df["qid"].astype(str)
print(len(qrels_df))
qrels_df.drop_duplicates(inplace=True, ignore_index=True)
print(len(qrels_df))

2303
2303


In [19]:
qrels_df = qrels_df.drop_duplicates(subset=["qid", "docno"], ignore_index=True, keep=False) # dropping qrels which have multiple relavances for a given (qid, docno) pair
len(qrels_df)

2303

In [20]:
qrels_df.to_csv("../data/kid-friend-en/en/qrels/qrels-relevance-graded.csv", index=False)
qrels_df.head()

,qid,q0,docno,relevance
0,1,0,77cdd0f6c3b04b5bbe79fac37c2ae63b,0
1,1,0,c8a2aa042145402a8c180ca6f467c575,1
2,1,0,d82e5e4f531841d49bcd6fe7bd8cdf8f,0
3,1,0,c8e3ba8d611441bebaf146001171af3a,1
4,1,0,44d7a109cd554b1cb44d2c3867ff7040,1


In [8]:
# binary topical relevance

qrels_df["relevance"] = [int(rel > 0) for rel in qrels_df["relevance"]]
qrels_df.to_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv", index=False)
qrels_df.head()

,qid,q0,docno,relevance
0,1,0,77cdd0f6c3b04b5bbe79fac37c2ae63b,0
1,1,0,c8a2aa042145402a8c180ca6f467c575,1
2,1,0,d82e5e4f531841d49bcd6fe7bd8cdf8f,0
3,1,0,c8e3ba8d611441bebaf146001171af3a,1
4,1,0,44d7a109cd554b1cb44d2c3867ff7040,1


In [9]:
qrels_df.dtypes

qid          object
q0           object
docno        object
relevance     int64
dtype: object

In [10]:
query_df.dtypes

qid            object
query          object
category       object
description    object
narrative      object
dtype: object

In [1]:
import pandas as pd

df = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
df["relevance"].value_counts()

relevance
1    1480
0     823
Name: count, dtype: int64

## Adding more document features

### Readability

In [2]:
# adding readability (spache-allen) scores of documents

import json
import pandas as pd
from spache_allen.formula import spache_allen
from tqdm import tqdm

lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus["readability"] = [spache_allen(text) for text in tqdm(corpus["snippet"], total=len(corpus))]
corpus.to_csv("../data/kid-friend-en/en/inputs/documents.csv", index=False)
print(len(corpus))
corpus.head()


100%|██████████| 2385/2385 [00:28<00:00, 84.02it/s]


2385


,docno,snippet,title,main_content,readability
0,6e421f1539b1457b853712d81be87743,WEBBTS (also Bangtan Boys; Korean: 방탄소년단 Bangt...,BTS (band) - Wikipedia,BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan ...,8.476857
1,1539268e3f1d41c9abcaa277b23f51d9,WEBMusic video by BTS performing Dynamite. (C)...,Before you go on to YouTube,YouTube\nA Google company\n\nBefore you go on ...,5.729500
2,93689ea1c0ec4272b41d7b259cb47890,Biography BTS is a Grammy-nominated South Kore...,Before you go on to YouTube,YouTube\nA Google company\n\nBefore you go on ...,6.051250
3,2e4321c5105840c8b6612f87dc4aba13,WEBBTS (Korean: 방탄소년단; RR: Bangtan Sonyeondan;...,BTS - Wikipedia,BTS (Korean방탄소년단; RRBangtan Sonyeondan; lit. B...,6.293952
4,3d21474fff4e406c9db2984d5edf647b,"WEBTo avoid this, cancel and sign in to YouTub...",BTS (방탄소년단) 'Yet To Come (The Most Beautiful M...,"So, Who is missing BTS badly. Let's stay stron...",5.167517


In [3]:
import pandas as pd

df = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")
len(df)

2385

In [4]:
import pandas as pd
import json

lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
len(corpus.loc[corpus["main_content"]==""]), len(corpus.loc[(corpus["main_content"]=="")&(corpus["snippet"]=="")]), len(corpus.loc[(corpus["main_content"]=="")&(corpus["snippet"]=="")&(corpus["title"]=="")])

(187, 15, 7)

In [5]:
corpus_df = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")
corpus_df.loc[(corpus_df["readability"]<0)&(corpus_df["main_content"].isna()==False)]

,docno,snippet,title,main_content,readability
1144,b0687fecdf9d477e833963defdf74e10,www.klimakleber.org,NaN,"climateglue.org\nis parked free, courtesy of G...",-1.0
1711,9fb79a165532437fb5d814a8a0784991,NaN,지민 (Jimin) 'Who' Official MV - YouTube,매일매일 수십번을 들어도 넘 좋은 WHO 진짜 고마워 지민 언제나 행복하게 기다리고...,-1.0
1712,5240a343b77c4428ba7d87bd092d02f6,NaN,NaN,BTS member Jin finishes military service - ban...,-1.0
1713,10727b9260ea4250a3576209aa40de2e,NaN,Amazon.com,Enter the characters below\n\nWe ask for your ...,-1.0
1714,3cca2583656a4c528dcc6b8213b5bdd7,NaN,BTS (Bangtan Boys) | BTS Wiki | Fandom,BTS Wiki\nAdvertisement\n\nBTS short info[]\n\...,-1.0
...,...,...,...,...,...
1952,9b416a567f5b4bb0bc2ce99cd2ab1e25,NaN,All about the first period during puberty | o.b.®,The first period\n\nWomen laugh and make faces...,-1.0
1953,3fcf27e780f147ae96e799b5e3832615,NaN,\n How do I know when I'm getting my period? ...,How do I feel that I am getting my period?\n\n...,-1.0
1962,3a667ab4ded14e8891d0180e2944a1bb,NaN,planet schule: How do breasts change during pu...,planet schule: How do breasts change during pu...,-1.0
1973,ae64d77ebaf74ee1839dc1557defa3a3,NaN,Recognize if a boy likes you,Magazine\n\nOverview of\nFilter:\n - icon30\n...,-1.0


In [6]:
corpus_df.loc[(corpus_df["readability"]<0)]

,docno,snippet,title,main_content,readability
1144,b0687fecdf9d477e833963defdf74e10,www.klimakleber.org,NaN,"climateglue.org\nis parked free, courtesy of G...",-1.0
1181,161338fc0d0c4385a973f9d861940566,NaN,Log in to X / X,NaN,-1.0
1709,c83695f0502b486086d939f3d15d9478,NaN,BTS (방탄소년단) 'Dynamite' Official MV - YouTube,NaN,-1.0
1711,9fb79a165532437fb5d814a8a0784991,NaN,지민 (Jimin) 'Who' Official MV - YouTube,매일매일 수십번을 들어도 넘 좋은 WHO 진짜 고마워 지민 언제나 행복하게 기다리고...,-1.0
1712,5240a343b77c4428ba7d87bd092d02f6,NaN,NaN,BTS member Jin finishes military service - ban...,-1.0
...,...,...,...,...,...
1962,3a667ab4ded14e8891d0180e2944a1bb,NaN,planet schule: How do breasts change during pu...,planet schule: How do breasts change during pu...,-1.0
1973,ae64d77ebaf74ee1839dc1557defa3a3,NaN,Recognize if a boy likes you,Magazine\n\nOverview of\nFilter:\n - icon30\n...,-1.0
1979,c2a251b89e224009ac1765bf4dd1d395,NaN,10 cute things boys LOVE when girls do them! -...,NaN,-1.0
1984,333fdd2f08234b25a2a61b44470cad01,NaN,What are 5 subtle signs that a girl likes you?...,Sort\n\nShe touches you seemingly at random wh...,-1.0


### Objectivity

In [8]:
import json
import pandas as pd
from utils import get_obj_likelihood
from make_predictions import generate_predictions
from tqdm import tqdm

lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus["obj_prob"] = get_obj_likelihood(list(corpus["snippet"]))
corpus = corpus[["docno","obj_prob"]]

corpus_df = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")
corpus_df = corpus_df.merge(corpus, on="docno")
corpus_df.to_csv("../data/kid-friend-en/en/inputs/documents.csv", index=False)
corpus_df.head()

# generate_predictions()

Device set to use cuda:0


,docno,snippet,title,main_content,readability,obj_prob
0,6e421f1539b1457b853712d81be87743,WEBBTS (also Bangtan Boys; Korean: 방탄소년단 Bangt...,BTS (band) - Wikipedia,BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan ...,8.476857,0.876282
1,1539268e3f1d41c9abcaa277b23f51d9,WEBMusic video by BTS performing Dynamite. (C)...,Before you go on to YouTube,YouTube\nA Google company\n\nBefore you go on ...,5.729500,0.846696
2,93689ea1c0ec4272b41d7b259cb47890,Biography BTS is a Grammy-nominated South Kore...,Before you go on to YouTube,YouTube\nA Google company\n\nBefore you go on ...,6.051250,0.895289
3,2e4321c5105840c8b6612f87dc4aba13,WEBBTS (Korean: 방탄소년단; RR: Bangtan Sonyeondan;...,BTS - Wikipedia,BTS (Korean방탄소년단; RRBangtan Sonyeondan; lit. B...,6.293952,0.876808
4,3d21474fff4e406c9db2984d5edf647b,"WEBTo avoid this, cancel and sign in to YouTub...",BTS (방탄소년단) 'Yet To Come (The Most Beautiful M...,"So, Who is missing BTS badly. Let's stay stron...",5.167517,0.615062


In [5]:
# import numpy as np

# lines = []
# with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
#     lines = f.read().splitlines()

# line_dicts = [json.loads(line) for line in lines]
# corpus = pd.DataFrame(line_dicts)

# np.argmax([len(text) for text in corpus['main_content']])

np.int64(1100)

In [6]:
# corpus.iloc[1100]

docno                            a5619809b4a34347a8cf77a6b8b72211
snippet         A coral reef is an underwater ecosystem charac...
title                                      Coral reef - Wikipedia
main_content    Jump to content\n\nCoral reef\n\nPage semi-pro...
Name: 1100, dtype: object

### Educational Alignment

In [9]:
import json
import pandas as pd
from tqdm import tqdm
from utils import get_edu_value

lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus["edu_val"] = [get_edu_value(text) for text in tqdm(corpus["snippet"], total=len(corpus))]

corpus = corpus[["docno","edu_val"]]
corpus_df = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")
corpus_df = corpus_df.merge(corpus, on="docno")
corpus_df.to_csv("../data/kid-friend-en/en/inputs/documents.csv", index=False)
corpus_df.head()


100%|██████████| 2385/2385 [32:15<00:00,  1.23it/s]


,docno,snippet,title,main_content,readability,obj_prob,edu_val
0,6e421f1539b1457b853712d81be87743,WEBBTS (also Bangtan Boys; Korean: 방탄소년단 Bangt...,BTS (band) - Wikipedia,BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan ...,8.476857,0.876282,1
1,1539268e3f1d41c9abcaa277b23f51d9,WEBMusic video by BTS performing Dynamite. (C)...,Before you go on to YouTube,YouTube\nA Google company\n\nBefore you go on ...,5.729500,0.846696,0
2,93689ea1c0ec4272b41d7b259cb47890,Biography BTS is a Grammy-nominated South Kore...,Before you go on to YouTube,YouTube\nA Google company\n\nBefore you go on ...,6.051250,0.895289,1
3,2e4321c5105840c8b6612f87dc4aba13,WEBBTS (Korean: 방탄소년단; RR: Bangtan Sonyeondan;...,BTS - Wikipedia,BTS (Korean방탄소년단; RRBangtan Sonyeondan; lit. B...,6.293952,0.876808,1
4,3d21474fff4e406c9db2984d5edf647b,"WEBTo avoid this, cancel and sign in to YouTub...",BTS (방탄소년단) 'Yet To Come (The Most Beautiful M...,"So, Who is missing BTS badly. Let's stay stron...",5.167517,0.615062,0


## Creating other versions of ground truth

### Relevance + Readability

In [10]:
# binary readability relevance: same query-doc pairs as qrels-relevance-binary.txt 
# query-doc pair assigned 1 if relevant and readable; 0 otherwise

from tqdm import tqdm
import pandas as pd

# nltk.download('all', halt_on_error=False)
corpus = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")
qrels_df = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
read_relevance = []
issue_docs = []

for _, row in tqdm(qrels_df.iterrows(), total=len(qrels_df)):
    qid = row["qid"]
    docid = row["docno"]
    relevance = row["relevance"]
    readability = list(corpus.loc[corpus["docno"]==docid]["readability"])[0]
    if readability < 0: # removing query-doc pairs where the readability score is invalid (i.e., the score < 0)
        issue_docs.append(docid)
        continue
    else:
        read_relevance.append([qid, "q0", docid, relevance*int(int(readability)<=4)])

qrels_readability_df = pd.DataFrame(read_relevance, columns=["qid", "q0", "docno", "relevance"])
qrels_readability_df.to_csv("../data/kid-friend-en/en/qrels/qrels-readability-binary.csv", index=False)
qrels_readability_df


100%|██████████| 2303/2303 [00:00<00:00, 2876.85it/s]


,qid,q0,docno,relevance
0,1,q0,77cdd0f6c3b04b5bbe79fac37c2ae63b,0
1,1,q0,c8a2aa042145402a8c180ca6f467c575,0
2,1,q0,d82e5e4f531841d49bcd6fe7bd8cdf8f,0
3,1,q0,c8e3ba8d611441bebaf146001171af3a,0
4,1,q0,44d7a109cd554b1cb44d2c3867ff7040,0
...,...,...,...,...
2211,50,q0,b7646dcb4b134afbacbe9a61a4ef914e,1
2212,50,q0,bc5f268885cb41379f06ced62f96adbb,0
2213,50,q0,c5840e5e858d4baa97ecf216dda4e6c3,0
2214,50,q0,bf5f2ac9b4454a79bd6848f10314bbee,0


In [11]:
assert (len(qrels_df)-len(qrels_readability_df))==len(issue_docs)

doc_issues = [x for x in list(qrels_df["docno"].unique()) if x not in list(qrels_readability_df["docno"].unique())]
print(len(doc_issues), len(set(issue_docs)))
for docno in doc_issues:
    print(docno, list(corpus.loc[corpus["docno"]==docno]["readability"])[0])

81 81
3cca2583656a4c528dcc6b8213b5bdd7 -1.0
ec742a5f53bf4c6db7aafdc361afbf61 -1.0
69153b48fc8a49d0b25288ff437c3710 -1.0
02cd6a52f92847d491ef13d36f42b2a6 -1.0
2357bd4bfba440cfbe6d86c66211ea63 -1.0
ad5c1a75ba0b47e58fcdfef8471a2cf7 -1.0
774e9cd59abb48d3a2687f0dc25f7156 -1.0
815e9f8e2ccd441491e7b1c1f2a05d49 -1.0
e800b727c5ff4395900c3078ed2752e6 -1.0
71b8acf3a6164762a2b044e8e763f05f -1.0
11c4c5258e8945309209c7156c24d3e0 -1.0
8cccf65c62d4458cb6934a2c678cf630 -1.0
527da37320224a27a37c93f125731db9 -1.0
e8653177217b4dbda0e9371de924675a -1.0
3b426dd868e1458eacc3c9fe8fe489e5 -1.0
fa974baf472e4443abafc3523d6e58fb -1.0
340c35086579401aa090a995e8cc6fb8 -1.0
bcb4bb8d25ab482b81e9fe51d0824092 -1.0
011f2d15eaf24cae9fa91666fd108458 -1.0
785d82f6d995495baac1586e2f92e7ba -1.0
3cd906efb7a24dbfade7214b87ebf365 -1.0
102463237fc9478ea8c107772539cdf9 -1.0
75e3dd963a054efb9aa73db8c9480502 -1.0
849bfea1e63e415cb5464341d83b9fc5 -1.0
a229b0bd3e7742be8f69820b3bc993f8 -1.0
2529ae2c690d4294b307f9b05b25f26d -1.0
6574ea

### Objectivity + Relevance

In [13]:
import json
import pandas as pd
from utils import pred_obj
from tqdm import tqdm

lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus["obj_prob"] = pred_obj(list(corpus["snippet"]))

# corpus.head()

qrels_df = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
obj_relevance = []
issue_docs = []

for _, row in tqdm(qrels_df.iterrows(), total=len(qrels_df)):
    qid = row["qid"]
    docid = row["docno"]
    relevance = row["relevance"]
    objectivity = list(corpus.loc[corpus["docno"]==docid]["obj_prob"])[0]
    obj_relevance.append([qid, "q0", docid, relevance*objectivity])

qrels_objectivity_df = pd.DataFrame(obj_relevance, columns=["qid", "q0", "docno", "relevance"])
qrels_objectivity_df.to_csv("../data/kid-friend-en/en/qrels/qrels-objectivity-binary.csv", index=False)
qrels_objectivity_df

Device set to use cuda:0

100%|██████████| 2303/2303 [00:00<00:00, 2941.39it/s]


,qid,q0,docno,relevance
0,1,q0,77cdd0f6c3b04b5bbe79fac37c2ae63b,0
1,1,q0,c8a2aa042145402a8c180ca6f467c575,1
2,1,q0,d82e5e4f531841d49bcd6fe7bd8cdf8f,0
3,1,q0,c8e3ba8d611441bebaf146001171af3a,1
4,1,q0,44d7a109cd554b1cb44d2c3867ff7040,1
...,...,...,...,...
2298,50,q0,b7646dcb4b134afbacbe9a61a4ef914e,0
2299,50,q0,bc5f268885cb41379f06ced62f96adbb,0
2300,50,q0,c5840e5e858d4baa97ecf216dda4e6c3,0
2301,50,q0,bf5f2ac9b4454a79bd6848f10314bbee,0


### Educational Alignment + Relevance

In [12]:
from tqdm import tqdm
import pandas as pd

corpus = pd.read_csv("../data/kid-friend-en/en/inputs/documents.csv")
qrels_df = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
edu_relevance = []
issue_docs = []

for _, row in tqdm(qrels_df.iterrows(), total=len(qrels_df)):
    qid = row["qid"]
    docid = row["docno"]
    relevance = row["relevance"]
    edu_val = list(corpus.loc[corpus["docno"]==docid]["edu_val"])[0]
    edu_relevance.append([qid, "q0", docid, relevance*int(edu_val>=3)])

qrels_edu_df = pd.DataFrame(edu_relevance, columns=["qid", "q0", "docno", "relevance"])
qrels_edu_df.to_csv("../data/kid-friend-en/en/qrels/qrels-edu-binary.csv", index=False)
qrels_edu_df


100%|██████████| 2303/2303 [00:00<00:00, 2875.58it/s]


,qid,q0,docno,relevance
0,1,q0,77cdd0f6c3b04b5bbe79fac37c2ae63b,0
1,1,q0,c8a2aa042145402a8c180ca6f467c575,0
2,1,q0,d82e5e4f531841d49bcd6fe7bd8cdf8f,0
3,1,q0,c8e3ba8d611441bebaf146001171af3a,0
4,1,q0,44d7a109cd554b1cb44d2c3867ff7040,0
...,...,...,...,...
2298,50,q0,b7646dcb4b134afbacbe9a61a4ef914e,0
2299,50,q0,bc5f268885cb41379f06ced62f96adbb,0
2300,50,q0,c5840e5e858d4baa97ecf216dda4e6c3,0
2301,50,q0,bf5f2ac9b4454a79bd6848f10314bbee,0
